# Bao 局面相転移候補分析

`pilot-v2` study version `0.4.1` の正式成果物を使い、形式的転移、戦略的転移候補、3/5 ply の持続性、7 ply 早期終局を除外した感度分析を実行する。分析ロジックの正本は `tools/experiments/analyze-phase-transition-pilot.py` とする。

## 1. 入力配置

Colab の `/content/pilot-v2/` に `observations.jsonl`、`games.json`、`manifest.json` を配置する。pilot-v1 比較も行う場合は `/content/pilot/games.json` を配置する。

In [ ]:
from pathlib import Path

PILOT_V2 = Path('/content/pilot-v2')
for name in ['observations.jsonl', 'games.json', 'manifest.json']:
    assert (PILOT_V2 / name).exists(), f'Missing: {PILOT_V2 / name}'
print('pilot-v2 inputs found')

## 2. リポジトリ取得と分析実行

In [ ]:
!git clone -q https://github.com/nkkmd/bao-la-kiswahili-game.git /content/bao-la-kiswahili-game || true
!python /content/bao-la-kiswahili-game/tools/experiments/analyze-phase-transition-pilot.py \
  --input /content/pilot-v2 \
  --output /content/phase-transition-analysis

## 3. 出力確認

- `analysis-summary.json`: 全100局と早期終局除外98局の感度比較、形式的イベント件数、閾値
- `transition-candidates.csv`: 持続性フィルターを通過した候補一覧

In [ ]:
import json
import pandas as pd

OUTPUT = Path('/content/phase-transition-analysis')
summary = json.loads((OUTPUT / 'analysis-summary.json').read_text())
candidates = pd.read_csv(OUTPUT / 'transition-candidates.csv')
display(pd.DataFrame(summary['samples']))
display(candidates.head(30))

## 4. 候補分布の可視化

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
candidates['normalized_ply'].hist(bins=20, ax=ax)
ax.set_title('Persistent transition candidates by normalized ply')
ax.set_xlabel('normalized ply')
ax.set_ylabel('candidate count')
plt.show()

## 解釈上の注意

候補スコアと閾値はパイロット探索用であり、正式な相転移認定値ではない。全100局と早期終局除外98局を併記し、新規 seed および別探索条件で再現した後に正式結論とする。